In [1]:
import sys
import os
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, avg, stddev, col, when
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
# 1. Environment Overrides
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/openjdk-17.jdk/Contents/Home"
os.environ["PYSPARK_PYTHON"] = sys.executable

# 2. Hardened Spark 4.1.1 Session for Gold Transformations
spark = SparkSession.builder \
    .appName("GoldTransformations") \
    .config("spark.jars.packages", (
        "org.apache.hadoop:hadoop-azure:3.4.1,"
        "com.microsoft.azure:azure-storage:8.6.6,"
        "io.delta:delta-spark_2.13:4.1.0"
    )) \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.azure.account.auth.type", "SharedKey") \
    .config("spark.hadoop.fs.abfss.impl", "org.apache.hadoop.fs.azurebfs.SecureAzureBlobFileSystem") \
    .getOrCreate()

# 3. Credentials
storage_account_name = "coinmarketcapapi"
container_name = "coin-market-cap-api-data"
storage_account_key = os.getenv("AZURE_STORAGE_KEY")

# Set the key on the DFS endpoint (Required for HNS)
spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

print(f"🚀 Spark {spark.version} Ready for Gold Layer Transformations!")

:: loading settings :: url = jar:file:/Users/as-mac-1250/crypto-market-intelligence/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/as-mac-1250/.ivy2.5.2/cache
The jars for the packages stored in: /Users/as-mac-1250/.ivy2.5.2/jars
org.apache.hadoop#hadoop-azure added as a dependency
com.microsoft.azure#azure-storage added as a dependency
io.delta#delta-spark_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-2b28c108-581e-4c50-9653-67ea67041345;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-azure;3.4.1 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found commons-logging#commons-logging;1.2 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.hadoop.thirdparty#hadoop-shaded-guava;1.3.0 in central
	found org.eclipse.jetty#jetty-util-aja

🚀 Spark 4.1.1 Ready for Gold Layer Transformations!


In [3]:
storage_account_name = "coinmarketcapapi"
container_name = "coin-market-cap-api-data"
storage_account_key = os.getenv("AZURE_STORAGE_KEY")

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_key)

# --- 2. PATH DEFINITIONS ---
silver_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/silver"
gold_base_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/gold"

silver_quotes_path = f"{silver_base_path}/crypto_quotes"
silver_global_path = f"{silver_base_path}/global_metrics"

gold_daily_summary_path = f"{gold_base_path}/daily_price_summary"
gold_volatility_path = f"{gold_base_path}/volatility_metrics"

In [4]:
# --- 2. DAILY PRICE SUMMARY & MOMENTUM ---
# Creating a table for day-to-day trends and price movements
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, avg, stddev, col

# --- 1. LOAD SILVER DATA ---
quotes_df = spark.read.format("delta").load(silver_quotes_path)
global_df = spark.read.format("delta").load(silver_global_path)

# --- 2. WINDOW SPECIFICATION ---
# Partition by coin and order by time to calculate rolling metrics
window_spec = Window.partitionBy("symbol").orderBy("event_timestamp")

# --- 3. DAILY PRICE SUMMARY & MOMENTUM ---
gold_daily_summary = quotes_df \
    .withColumn("prev_day_price", lag("price_usd", 1).over(window_spec)) \
    .withColumn("daily_return", 
        when(col("prev_day_price").isNull(), 0)
        .otherwise((col("price_usd") - col("prev_day_price")) / col("prev_day_price"))) \
    .withColumn("7d_moving_avg", avg("price_usd").over(window_spec.rowsBetween(-7, 0))) \
    .withColumn("14d_moving_avg", avg("price_usd").over(window_spec.rowsBetween(-14, 0))) \
    .withColumn("7d_avg_volume", avg("volume_24h").over(window_spec.rowsBetween(-7, 0))) \
    .withColumn("volume_spike", 
        when(col("volume_24h") > (col("7d_avg_volume") * 2), 1).otherwise(0))

# 4. Integrate Market Dominance
# We join with global metrics to see how much of the total market this specific coin represents
latest_global = global_df.select("event_timestamp", "total_market_cap")

gold_daily_summary = gold_daily_summary.join(latest_global, "event_timestamp", "left") \
    .withColumn("market_dominance_pct", (col("market_cap") / col("total_market_cap")) * 100)

gold_volatility_metrics = quotes_df \
    .withColumn("rolling_std", stddev("price_usd").over(window_spec.rowsBetween(-14, 0))) \
    .select("symbol", "rolling_std", "market_cap", "event_timestamp")

# --- 5. WRITE TO GOLD LAYER ---
gold_daily_summary.write.format("delta").mode("overwrite").save(gold_daily_summary_path)
gold_volatility_metrics.write.format("delta").mode("overwrite").save(gold_volatility_path)

print("✅ Gold Layer processing complete and saved to ADLS Gen2!")

26/04/16 20:24:47 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


✅ Gold Layer processing complete and saved to ADLS Gen2!


In [5]:
# --- 1. VALIDATE DAILY SUMMARY & MOMENTUM ---
print("📊 Checking Daily Summary & 7-Day Moving Averages:")
gold_daily_df = spark.read.format("delta").load(gold_daily_summary_path)
gold_daily_df.sort(col("event_timestamp").desc()).show(10)

# --- 2. VALIDATE VOLATILITY & DOMINANCE ---
print("📊 Checking Volatility Metrics & Market Cap:")
gold_vol_df = spark.read.format("delta").load(gold_volatility_path)
gold_vol_df.sort(col("event_timestamp").desc()).show(10)

📊 Checking Daily Summary & 7-Day Moving Averages:
+--------------------+-------+------+-----------------+-------------------+--------------------+--------------------+---------------------------+--------------+------------+-------------------+-------------------+--------------------+------------+----------------+--------------------+
|     event_timestamp|coin_id|symbol|             name|          price_usd|          volume_24h|          market_cap|silver_processing_timestamp|prev_day_price|daily_return|      7d_moving_avg|     14d_moving_avg|       7d_avg_volume|volume_spike|total_market_cap|market_dominance_pct|
+--------------------+-------+------+-----------------+-------------------+--------------------+--------------------+---------------------------+--------------+------------+-------------------+-------------------+--------------------+------------+----------------+--------------------+
|2026-04-16T12:09:...|   7278|  AAVE|             Aave| 106.50252156032039| 4.28212479363720

In [6]:
# KPI: Total Crypto Market Cap Trend (from Silver Global)
print("📈 Total Crypto Market Cap Trend:")
spark.read.format("delta").load(silver_global_path) \
    .select("event_timestamp", "total_market_cap") \
    .orderBy("event_timestamp") \
    .show(10)

📈 Total Crypto Market Cap Trend:
+--------------------+-----------------+
|     event_timestamp| total_market_cap|
+--------------------+-----------------+
|2026-04-16T12:10:...|2.529362757265E12|
+--------------------+-----------------+



In [7]:
# KPI: Top 5 Gainers (last 24 hours) 
print("🏆 Top 5 Gainers (Last 24 Hours):")
gold_daily_df.select("symbol", "daily_return") \
    .orderBy(col("daily_return").desc()) \
    .limit(5) \
    .show()

🏆 Top 5 Gainers (Last 24 Hours):
+------+------------+
|symbol|daily_return|
+------+------------+
|  AAVE|         0.0|
|   ADA|         0.0|
|  AERO|         0.0|
|  ALGO|         0.0|
|   APT|         0.0|
+------+------------+



In [8]:
# Feature Validation: Price vs Moving Average (BTC Example)
print("🔍 BTC Price vs 7-Day Moving Average:")
gold_daily_df.filter(col("symbol") == "BTC") \
    .select("event_timestamp", "price_usd", "7d_moving_avg") \
    .orderBy(col("event_timestamp").desc()) \
    .show(10)

🔍 BTC Price vs 7-Day Moving Average:


+--------------------+-----------------+-----------------+
|     event_timestamp|        price_usd|    7d_moving_avg|
+--------------------+-----------------+-----------------+
|2026-04-16T12:09:...|74650.23175439381|74650.23175439381|
+--------------------+-----------------+-----------------+



In [9]:
spark.stop()